# Run a Model x Benchmark Matrix

This notebook shows the smallest useful pattern for running real model wrappers against real benchmark wrappers. It uses `run_benchmark_matrix`, which writes one JSON result file per model/benchmark pair.

## 1. Runtime Setup

If this notebook is already inside the repo, skip the clone step and run the install step only if the runtime does not already have the dependencies.

In [ ]:
# Colab / RunPod setup. Uncomment when starting from a fresh runtime.
# !git clone https://github.com/samhatescoding/transformers.git
# %cd transformers
# !pip install -r requirements.txt

In [ ]:
from pathlib import Path
import json
import pandas as pd

from runners.benchmark_run import BenchmarkRun
from runners.execution import run_benchmark_matrix
from runners.model_run import ModelRun

OUTPUT_DIR = Path("results/notebook_matrix")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 2. Choose Real Models

Start with a small `max_new_tokens` value for label-style benchmarks. Loading two local vision-language models can require substantial VRAM, so reduce this list if the runtime is small.

In [ ]:
from models import LlavaGemma2B, Qwen25VL3B

models = [
    ModelRun.from_factory(
        name="llava-gemma-2b",
        factory=LlavaGemma2B,
        stream=False,
        max_new_tokens=16,
    ),
    ModelRun.from_factory(
        name="qwen2.5-vl-3b-instruct",
        factory=Qwen25VL3B,
        max_new_tokens=16,
    ),
]

## 3. Choose Real Benchmarks

`num_samples=2` keeps this quick while proving that the matrix works. Increase it once the setup is stable.

In [ ]:
from benchmarks import FashionMNISTBenchmark, GQABenchmark

benchmark_runs = [
    BenchmarkRun(benchmark=FashionMNISTBenchmark(streaming=True), num_samples=2),
    BenchmarkRun(benchmark=GQABenchmark(streaming=True), num_samples=2),
]

## 4. Run the Matrix

This runs every model in `models` against every benchmark in `benchmark_runs`. With the defaults above, it produces four result files.

In [ ]:
summaries = run_benchmark_matrix(
    models=models,
    benchmark_runs=benchmark_runs,
    output_dir=OUTPUT_DIR,
)

summary_df = pd.DataFrame(summaries)
summary_df

## 5. Inspect Results

Each result file contains the model name, benchmark name, per-sample predictions, correctness flags, and aggregate stats.

In [ ]:
result_files = sorted(OUTPUT_DIR.glob("*.json"))
result_files

In [ ]:
with result_files[0].open("r", encoding="utf-8") as f:
    first_result = json.load(f)

first_result.keys(), first_result["report"].keys()

In [ ]:
rows = []
for path in result_files:
    payload = json.loads(path.read_text(encoding="utf-8"))
    report = payload["report"]
    stats = report.get("stats", {})
    rows.append(
        {
            "model": payload["model"],
            "benchmark": payload["benchmark"],
            "samples": report.get("num_samples"),
            "success_count": stats.get("success_count"),
            "failure_count": stats.get("failure_count"),
            "wall_clock_seconds": stats.get("wall_clock_time_seconds"),
            "path": str(path),
        }
    )

pd.DataFrame(rows)

## 6. Swap in Different Models or Benchmarks

The only pieces you usually change are the two lists. Add or remove `ModelRun` and `BenchmarkRun` entries, then rerun the matrix cell.

In [ ]:
# Example alternatives:
# from benchmarks import Flickr30kBenchmark, VQAv2Benchmark
# from models import LlavaOnevisionQwen2_7B, GPT4
#
# models = [
#     ModelRun.from_factory("llava-onevision-qwen2-7b", LlavaOnevisionQwen2_7B, stream=False, max_new_tokens=32),
#     ModelRun.from_factory("gpt-4o", GPT4, max_new_tokens=32),
# ]
# benchmark_runs = [
#     BenchmarkRun(Flickr30kBenchmark(streaming=True), num_samples=5),
#     BenchmarkRun(VQAv2Benchmark(streaming=True), num_samples=5),
# ]